In [ ]:
!pip install -U bitsandbytes transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 145.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.6/536.6 kB 51.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6


# Task
Load and process a nested JSON file containing 'id', 'syllogism', 'validity', and 'plausibility' fields. Extract syllogism sentences, identify key concepts, map these concepts to logical placeholders (X, Y, Z), and transform each sentence into a standardized logical structure (e.g., 'All X are Y. No Z is X. Therefore no Z is Y.') using a predefined set of logical words.

## Load Nested JSON Data

### Subtask:
Load the data from the nested JSON file, which contains objects with 'id', 'syllogism', 'validity', and 'plausibility' fields.


In [ ]:
import json
import pandas as pd

# Use the correct relative path from the notebook's location to the data file
with open('data.json', 'r') as f:
    data = json.load(f)

df_syllogisms = pd.DataFrame(data)

print("DataFrame loaded successfully. Displaying the first 5 rows:")
print(df_syllogisms.head())

syllogism_sentences = df_syllogisms['syllogism'].tolist()

print(f"Extracted {len(syllogism_sentences)} syllogism sentences. Displaying the first 3:")
for i, sentence in enumerate(syllogism_sentences[:3]):
    print(f"- {sentence}")

In [ ]:
# Check GPU availability and configure PyTorch
import torch

print("=" * 80)
print("GPU CONFIGURATION")
print("=" * 80)
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Count: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Total Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"GPU Current Device: {torch.cuda.current_device()}")
else:
    print("WARNING: No GPU detected. Model will run on CPU (slow).")
print("=" * 80)

GPU CONFIGURATION
GPU Available: True
GPU Count: 1
GPU Name: NVIDIA A100-SXM4-40GB
GPU Total Memory: 42.47 GB
GPU Current Device: 0


In [ ]:
import torch
torch.cuda.empty_cache()

## prepare prompt

### Subtask:
Extract the syllogism sentences and create prompts for the Large Language Model. The prompt is made to handle both subject mapping and sentence simplification. To reduce hallucinations we give the model strict rules, and give for each result a reasoning field so it can do some Chain of Tought.


In [ ]:
def transform_syllogisms(groups_list):
    # Join groups with clear delimiters
    formatted_input = ""
    for i, group in enumerate(groups_list):
        group_split = group.split('. ')
        formatted_input += f"---\nGROUP {i+1}:\n"
        formatted_input += "\n".join(group_split) + "\n"

    return formatted_input
def make_prompt(sentences):

    formatted_input = transform_syllogisms(sentences)
    prompt = f"""You are an expert in syllogism linguistics and formal logic. Your task is to translate natural language sentences into their fundamental categorical propositions.

### Task
You will receive multiple groups of sentences. For each group, provide a mapping of concepts to letters and a list of simplified sentences.
The sentences should be simplified based on Aristotelian logic:
1. All X are Y (Universal Affirmative)
2. Some X are not Y (Particular Negative)
3. Some X are Y (Particular Affirmative)
4. No X are Y (Universal Negative)

### Rules
- **Per-Group Mapping:** Reset letter assignments (A, B, C...) for every new group.
- **Concept Extraction:** Extract exactly two concepts per sentence.
- **Normalization:** Strip quantifiers (all, some), negations (not, no), and copulas (is, are). You can use them to determine the simplified form.
- **Core Meaning:** Map different surface forms to the same concept if they are semantically identical in context (e.g., "those who breathe" and "breathers" → "Breathers").
- **Atribute Subgroups** Concepts can be changed by the presence of atributes. (eg. "gray hound" and "hound" are different concepts).
- **Logical Equivalences** "Not all X are Y" is the same as "Some X are not Y" / "All X are not Y" is the same as "No X are Y" / "Only X are Y" is the same as "All Y are X".
- **Symbolic Mapping:** Assign letters (A, B, C...) in order of first appearance. If a concept repeats, reuse the assigned letter.

### Processing Logic
1. List each sentence.
2. Identify concepts.
3. Determine the logical quantifier/relationship.
4. Check if either concept has appeared before.
5. Generate the final mapping and formal sentence.

Output structure:
{{
  "results": [
    {{
      "group_id": 1,
      "mapping": {{ "concept_1": "A" , "concept_2": "B", "concept_3": "C"}},
      "simplified_sentences": ["All A are B", "No B are C", "Some A are not C"],
      "reasoning": "Text with procces explanation"
    }}
  ]
}}

Input Data:
{formatted_input}
---"""
    return prompt


## run LLM

We run the LLM with batches of our data. This is the most efficient way to save on computation time and resources, with no effect to performance.


In [ ]:
from google import genai

client = genai.Client(api_key='insert api key')



In [ ]:

simplified_sentences = []
batches = [syllogism_sentences[i:i + 12] for i in range(0, len(syllogism_sentences), 12)]

for batch in batches[15:]:
    prompt = make_prompt(batch)
    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=prompt,
    )

    results = response.text
    dict_results = json.loads(results[8:-3])
    print(dict_results['results'])
    simplified_sentences+=dict_results['results']

simple_sentences = pd.DataFrame(simplified_sentences)['simplified_sentences']


[{'group_id': 1, 'mapping': {'Pentagons': 'A', 'Things with two sides': 'B', 'Triangles': 'C', 'Perfectly round things': 'D', 'Rectangles': 'E', 'Squares': 'F', 'Circles': 'G', 'Things with four corners': 'H', 'Lines': 'I', 'Three-dimensional cubes': 'J', 'Shapes': 'K', 'Flat things': 'L', 'Quadrilaterals': 'M'}, 'simplified_sentences': ['All A are B', 'Some C are D', 'Some E are F', 'All G are H', 'All I are J', 'No K are L', 'Some M are F', 'Some M are not E'], 'reasoning': "Standard quantifiers like 'all', 'any', and 'every' are mapped to 'All'. 'Some' and 'a portion' are mapped to 'Some'. 'There are no' is 'No'. Concept mapping treats 'quadrilaterals' as a distinct class from 'rectangles' and 'squares'."}, {'group_id': 2, 'mapping': {'Leopards': 'A', 'Fish': 'B', 'Panthers': 'C', 'Things made of wood': 'D', 'Cats': 'E', 'Felines': 'F', 'Jaguars': 'G', 'Reptiles': 'H', 'Tigers': 'I'}, 'simplified_sentences': ['Some A are B', 'All C are D', 'No E are F', 'All G are H', 'All I are E',

## deterministic validation

### Subtask:
Now that we have the simplified sentences we can use a deterministic algorithm to check if the conclusion in our syllogism is valid. We make a class that saves the premise relatioships and uses them to check the conclusion validity.


In [ ]:
#@title get validity subtask1

import re
class syllogism_relation:
    def __init__(self):
        self.objects = set()
        self.inclusion_edges = dict()
        self.inversed_inclusion_edges = dict()
        self.disjunction = set()
        self.overlap = set()
        self.non_inclusion = set()

    def add_relation(self, sentence):
        keywords = sentence.split()
        if re.match(r"^All (\w+) are (\w+)$", sentence):
            self.objects.add(keywords[1])
            self.objects.add(keywords[3])
            self.inclusion_edges[keywords[3]] = keywords[1]
            self.inversed_inclusion_edges[keywords[1]] = keywords[3]
        if re.match(r"^Some (\w+) are not (\w+)$", sentence):
            self.objects.add(keywords[4])
            self.objects.add(keywords[1])
            self.non_inclusion.add((keywords[1], keywords[4]))
        if re.match(r"^Some (\w+) are (\w+)$", sentence):
            self.objects.add(keywords[3])
            self.objects.add(keywords[1])
            self.overlap.add((keywords[1], keywords[3]))
        if re.match(r"^No (\w+) are (\w+)$", sentence):
            self.objects.add(keywords[3])
            self.objects.add(keywords[1])
            self.disjunction.add((keywords[1], keywords[3]))

    def conclude_relation(self, sentence):
        keywords = sentence.split()
        objA = keywords[1]
        objB = keywords[3]

        noninclusion = False
        overlap = False
        disjunction = False
        inclusion = False

        if re.match(r"^All (\w+) are (\w+)$", sentence):
            inclusion = True
        if re.match(r"^Some (\w+) are not (\w+)$", sentence):
            objB = keywords[4]
            noninclusion = True

        if re.match(r"^Some (\w+) are (\w+)$", sentence):
            overlap = True

        if re.match(r"^No (\w+) are (\w+)$", sentence):
            disjunction = True


        if (objA, objB) in self.disjunction or (objB, objA) in self.disjunction:
            if disjunction or noninclusion:
                return True

        if (objA, objB) in self.overlap or (objB, objA) in self.overlap:
            if overlap:
                return True

        if (objA, objB) in self.non_inclusion:
            if noninclusion:
                return True

        if noninclusion:
            for obj in self.objects:
                if ((objA, obj) in self.disjunction or (obj, objA) in self.disjunction) and ((objB,obj) in self.overlap or (obj, objB) in self.overlap):
                    return True
                if ((objA, obj) in self.overlap or (obj, objA) in self.overlap) and ((objB,obj) in self.disjunction or (obj, objB) in self.disjunction):
                    return True
        obj_move = objA
        while obj_move in self.inversed_inclusion_edges.keys():
            obj_move = self.inversed_inclusion_edges[obj_move]
            if (obj_move, objB) in self.disjunction or (objB, obj_move) in self.disjunction:
                if disjunction or noninclusion:
                    return True

        obj_move = objA
        while obj_move in self.inclusion_edges.keys():
            obj_move = self.inclusion_edges[obj_move]
            if (obj_move, objB) in self.non_inclusion or (obj_move,objB) in self.disjunction or (objB, obj_move) in self.disjunction:
                if noninclusion:
                    return True
            if (obj_move, objB) in self.overlap or (objB, obj_move) in self.overlap:
                if overlap:
                    return True
        obj_core = obj_move

        obj_move = objB

        while obj_move in self.inversed_inclusion_edges.keys():
            obj_move = self.inversed_inclusion_edges[obj_move]
            if (obj_move, objA) in self.disjunction or (objA, obj_move) in self.disjunction:
                if disjunction or noninclusion:
                    return True
            if (objA, obj_move) in self.non_inclusion:
                if noninclusion:
                    return True

        obj_move = objB
        while obj_move in self.inclusion_edges.keys():
            obj_move = self.inclusion_edges[obj_move]
            if (obj_move, objA) in self.overlap or (objA, obj_move) in self.overlap:
                if overlap:
                    return True
            if obj_move == objA:
                if inclusion or overlap:
                    return True

        if obj_move == obj_core:
            if overlap:
                return True

        # print(objB,self.disjunction, self.inclusion_edges)
        return False

def get_validity(sentences):
    syll = syllogism_relation()
    for sentence in sentences[:-1]:
        syll.add_relation(sentence)
    return syll.conclude_relation(sentences[-1])


In [ ]:
#@title get validity subtask 2
import re
class syllogism_relation:
    def __init__(self):
        self.objects = set()
        self.inclusion_edges = dict()
        self.inversed_inclusion_edges = dict()
        self.disjunction = dict()
        self.overlap = dict()
        self.non_inclusion = dict()

    def add_relation(self, sentence, id):
        keywords = sentence.split()
        if re.match(r"^All (\w+) are (\w+)$", sentence):
            self.objects.add(keywords[1])
            self.objects.add(keywords[3])
            self.inclusion_edges[keywords[3]] = (keywords[1],id)
            self.inversed_inclusion_edges[keywords[1]] = (keywords[3],id)
        if re.match(r"^Some (\w+) are not (\w+)$", sentence):
            self.objects.add(keywords[4])
            self.objects.add(keywords[1])
            self.non_inclusion[(keywords[1], keywords[4])]=id
        if re.match(r"^Some (\w+) are (\w+)$", sentence):
            self.objects.add(keywords[3])
            self.objects.add(keywords[1])
            self.overlap[(keywords[1], keywords[3])]=id
        if re.match(r"^No (\w+) are (\w+)$", sentence):
            self.objects.add(keywords[3])
            self.objects.add(keywords[1])
            self.disjunction[(keywords[1], keywords[3])]=id

    def conclude_relation(self, sentence):
        keywords = sentence.split()
        objA = keywords[1]
        objB = keywords[3]

        noninclusion = False
        overlap = False
        disjunction = False
        inclusion = False

        if re.match(r"^All (\w+) are (\w+)$", sentence):
            inclusion = True
        if re.match(r"^Some (\w+) are not (\w+)$", sentence):
            objB = keywords[4]
            noninclusion = True
        if re.match(r"^Some (\w+) are (\w+)$", sentence):
            overlap = True
        if re.match(r"^No (\w+) are (\w+)$", sentence):
            disjunction = True

        key1 = (objA, objB)
        key2 = (objB, objA)

        if key1 in self.disjunction.keys() or key2 in self.disjunction.keys():
            if disjunction or noninclusion:
                rel_id = self.disjunction[key1] if key1 in self.disjunction.keys() else self.disjunction[key2]
                return True, [rel_id]

        if key1 in self.overlap.keys() or key2 in self.overlap.keys():
            if overlap:
                rel_id = self.overlap[key1] if key1 in self.overlap.keys() else self.overlap[key2]
                return True, rel_id

        if key1 in self.non_inclusion.keys():
            if noninclusion:
                return True, self.non_inclusion[key1]

        if noninclusion:
            for obj in self.objects:
                keyA1 = (objA, obj)
                keyA2 = (obj, objA)
                keyB1 = (objB,obj)
                keyB2 = (obj, objB)
                if (keyA1 in self.disjunction.keys() or keyA2 in self.disjunction.keys()) and (keyB1 in self.overlap.keys() or keyB2 in self.overlap.keys()):
                    rel_id1 = self.disjunction[keyA1] if keyA1 in self.disjunction.keys() else self.disjunction[keyA2]
                    rel_id2 = self.overlap[keyB1] if keyB1 in self.overlap.keys() else self.overlap[keyB2]
                    return True, [rel_id1, rel_id2]
                if (keyA1 in self.overlap.keys() or keyA2 in self.overlap.keys()) and (keyB1 in self.disjunction.keys() or keyB2 in self.disjunction.keys()):
                    rel_id1 = self.overlap[keyA1] if keyA1 in self.overlap.keys() else self.overlap[keyA2]
                    rel_id2 = self.disjunction[keyB1] if keyB1 in self.disjunction.keys() else self.disjunction[keyB2]
                    return True, [rel_id1, rel_id2]



        obj_move = objA
        rel_list = []
        while obj_move in self.inversed_inclusion_edges.keys():
            rel_list.append(self.inversed_inclusion_edges[obj_move][1])
            obj_move = self.inversed_inclusion_edges[obj_move][0]
            key1 = (obj_move, objB)
            key2 = (objB, obj_move)
            if obj_move == objB:
                if inclusion or overlap:
                    return True, rel_list
            if key1 in self.disjunction or key2 in self.disjunction:
                if disjunction or noninclusion:
                    rel_id = self.disjunction[key1] if key1 in self.disjunction.keys() else self.disjunction[key2]
                    rel_list.append(rel_id)
                    return True, rel_list

        obj_move = objA
        rel_list = []
        while obj_move in self.inclusion_edges.keys():
            rel_list.append(self.inclusion_edges[obj_move][1])
            obj_move = self.inclusion_edges[obj_move][0]
            key1 = (obj_move, objB)
            key2 = (objB, obj_move)
            if obj_move == objB:
                if overlap:
                    return True, rel_list
            if key1 in self.non_inclusion or key1 in self.disjunction or key2 in self.disjunction:
                if noninclusion:
                    if key1 in self.non_inclusion.keys():
                        rel_id = self.non_inclusion[key1]
                    else:
                        rel_id = self.disjunction[key1] if key1 in self.disjunction.keys() else self.disjunction[key2]
                    rel_list.append(rel_id)
                    return True, rel_list
            if key1 in self.overlap or key2 in self.overlap:
                if overlap:
                    rel_id = self.overlap[key1] if key1 in self.overlap.keys() else self.overlap[key2]
                    rel_list.append(rel_id)
                    return True, rel_list
        obj_core = obj_move
        rel_list_core = rel_list

        obj_move = objB
        rel_list = []
        while obj_move in self.inversed_inclusion_edges.keys():
            rel_list.append(self.inversed_inclusion_edges[obj_move][1])
            obj_move = self.inversed_inclusion_edges[obj_move][0]
            key1 = (obj_move, objA)
            key2 = (objA, obj_move)
            if key1 in self.disjunction or key2 in self.disjunction:
                if disjunction or noninclusion:
                    rel_id = self.disjunction[key1] if key1 in self.disjunction.keys() else self.disjunction[key2]
                    rel_list.append(rel_id)
                    return True, rel_list
            if key2 in self.non_inclusion:
                if noninclusion:
                    rel_list.append(self.non_inclusion[key2])
                    return True, rel_list
            if obj_move == objA:
                if overlap:
                    return True, rel_list


        obj_move = objB
        rel_list = []
        while obj_move in self.inclusion_edges.keys():
            rel_list.append(self.inclusion_edges[obj_move][1])
            obj_move = self.inclusion_edges[obj_move][0]
            key1 = (obj_move, objA)
            key2 = (objA, obj_move)
            if key1 in self.overlap or key2 in self.overlap:
                if overlap:
                    rel_id = self.overlap[key1] if key1 in self.overlap.keys() else self.overlap[key2]
                    rel_list.append(rel_id)
                    return True, rel_list
            if obj_move == objA:
                if inclusion or overlap:
                    return True, rel_list

        if obj_move == obj_core:
            if overlap:
                return True, rel_list_core+rel_list

        # print(objB,self.disjunction, self.inclusion_edges)
        return (False, [])



def get_validity(sentences):
    syll = syllogism_relation()
    for id, sentence in enumerate(sentences[:-1]):
        syll.add_relation(sentence, id)
    # print(syll.inclusion_edges)
    # print(syll.inversed_inclusion_edges)
    return syll.conclude_relation(sentences[-1])




In [ ]:
validities = []
for simp in simple_sentences:
    validities.append(get_validity(simp))

## check score

### Subtask:
review and output the validities


In [ ]:
ids =df_syllogisms['id'].to_list()

In [ ]:
competition_results = []
for id, val in zip(ids,validities):
    competition_results.append({'id':id,'validity':val[0], 'relevant_premises':val[1]})

In [ ]:
with open('predictionsjson', 'w') as f:
    json.dump(competition_results,f, indent=4)

In [ ]:
#@title s1 scoring

import math
import sys
from typing import List, Dict, Any, Tuple, Optional

# --- 1. CORE FUNCTIONS FOR ACCURACY AND BIAS CALCULATION ---

def calculate_accuracy(
    ground_truth_list: List[Dict[str, Any]],
    predictions_list: List[Dict[str, Any]],
    metric_name: str,
    prediction_key: str,
    plausibility_filter: Optional[bool] = None
) -> Tuple[float, int, int]:
    """
    Calculates the accuracy of 'validity' predictions against ground truth labels,
    with an optional filter based on ground truth 'plausibility'.
    """
    gt_map = {item['id']: item for item in ground_truth_list}

    correct_predictions = 0
    total_predictions = 0

    for pred_item in predictions_list:
        item_id = pred_item['id']

        if item_id in gt_map:
            gt_item = gt_map[item_id]

            gt_plausibility = gt_item.get('plausibility')
            if plausibility_filter is not None and gt_plausibility != plausibility_filter:
                continue

            if metric_name in gt_item and prediction_key in pred_item:
                true_label = gt_item[metric_name]
                predicted_label = pred_item[prediction_key]

                if isinstance(true_label, bool) and isinstance(predicted_label, bool):
                    total_predictions += 1
                    if true_label == predicted_label:
                        correct_predictions += 1

    if total_predictions == 0:
        return 0.0, 0, 0

    accuracy = (correct_predictions / total_predictions) * 100
    return accuracy, correct_predictions, total_predictions


def calculate_subgroup_accuracy(
    gt_map: Dict[str, Any],
    predictions_list: List[Dict[str, Any]],
    gt_validity: bool,
    gt_plausibility: bool
) -> Tuple[float, int, int]:
    """
    Calculates accuracy for a specific subgroup defined by ground truth validity AND plausibility.
    """
    correct_predictions = 0
    total_predictions = 0

    for pred_item in predictions_list:
        item_id = pred_item['id']
        if item_id in gt_map:
            gt_item = gt_map[item_id]

            if gt_item.get('validity') == gt_validity and gt_item.get('plausibility') == gt_plausibility:
                if 'validity' in gt_item and 'validity' in pred_item:
                    true_label = gt_item['validity']
                    predicted_label = pred_item['validity']

                    if isinstance(true_label, bool) and isinstance(predicted_label, bool):
                        total_predictions += 1
                        if true_label == predicted_label:
                            correct_predictions += 1

    if total_predictions == 0:
        return 0.0, 0, 0

    accuracy = (correct_predictions / total_predictions) * 100
    return accuracy, correct_predictions, total_predictions


def calculate_content_effect_bias(accuracies: Dict[str, float]) -> Dict[str, float]:
    """
    Calculates the content effect bias metrics as defined.
    """

    acc_plausible_valid = accuracies.get('acc_plausible_valid', 0.0)
    acc_implausible_valid = accuracies.get('acc_implausible_valid', 0.0)
    acc_plausible_invalid = accuracies.get('acc_plausible_invalid', 0.0)
    acc_implausible_invalid = accuracies.get('acc_implausible_invalid', 0.0)

    # 1. content_effect_intra_validity_label
    intra_valid_diff = abs(acc_plausible_valid - acc_implausible_valid)
    intra_invalid_diff = abs(acc_plausible_invalid - acc_implausible_invalid)
    content_effect_intra_validity_label = (intra_valid_diff + intra_invalid_diff) / 2.0

    # 2. content_effect_inter_validity_label
    inter_plausible_diff = abs(acc_plausible_valid - acc_plausible_invalid)
    inter_implausible_diff = abs(acc_implausible_valid - acc_implausible_invalid)
    content_effect_inter_validity_label = (inter_plausible_diff + inter_implausible_diff) / 2.0

    # 3. tot_content_effect
    tot_content_effect = (content_effect_intra_validity_label + content_effect_inter_validity_label) / 2.0

    return {
        'content_effect_intra_validity_label': content_effect_intra_validity_label,
        'content_effect_inter_validity_label': content_effect_inter_validity_label,
        'tot_content_effect': tot_content_effect
    }

def calculate_smooth_combined_metric(overall_accuracy: float, total_content_effect: float) -> float:
    """
    Computes a smooth combined score using the natural logarithm.
    Formula: accuracy / (1 + ln(1 + content_effect))
    """
    if total_content_effect < 0:
        return 0.0

    log_penalty = math.log(1 + total_content_effect)
    combined_smooth_score = overall_accuracy / (1 + log_penalty)
    return combined_smooth_score


def run_full_scoring(reference_data_path: str, prediction_path: str, output_path: str):
    """
    Runs the full analysis pipeline for a single submission and writes results to the output path.
    """

    try:
        # 1. Load data from the provided file paths
        with open(reference_data_path, 'r') as f:
            ground_truth = json.load(f)

        with open(prediction_path, 'r') as f:
            predictions = json.load(f)

        # Check that the examples in ground_truth are all covered in predictions
        gt_ids = set([example["id"] for example in ground_truth])
        pd_ids = set([example["id"] for example in predictions])
        diff = len(gt_ids.difference(pd_ids))

        if diff != 0:
            print(f"Error: not all the examples in ground truth have a corresponding prediction", file=sys.stderr)
            final_results = {'accuracy': 0.0, 'f1_premises': 0.0, 'combined_score': 0.0}
            # --- Write Results ---
            try:
                with open(output_path, 'w') as f:
                    json.dump(final_results, f)
                    print(f"Scoring complete. Results written to {output_path}")
            except Exception as e:
                print(f"Error writing final results to file: {e}", file=sys.stderr)

            return

    except FileNotFoundError as e:
        print(f"Error: Required file not found. {e}", file=sys.stderr)
        final_results = {'accuracy': 0.0, 'content_effect': 100.0, 'combined_score': 0.0}
    except json.JSONDecodeError as e:
        print(f"Error: Invalid JSON format in submission or reference file. {e}", file=sys.stderr)
        final_results = {'accuracy': 0.0, 'content_effect': 100.0, 'combined_score': 0.0}
    except Exception as e:
        print(f"An unexpected error occurred during file loading: {e}", file=sys.stderr)
        final_results = {'accuracy': 0.0, 'content_effect': 100.0, 'combined_score': 0.0}

    else:
        gt_map = {item['id']: item for item in ground_truth}

        # --- 2. Calculate Overall Accuracy ---
        common_args = {
            'ground_truth_list': ground_truth,
            'predictions_list': predictions,
            'metric_name': 'validity',
            'prediction_key': 'validity'
        }
        overall_acc, _, _ = calculate_accuracy(**common_args, plausibility_filter=None)

        # --- 3. Calculate Conditional Accuracies for Bias Metrics ---
        acc_plausible_valid, _, _ = calculate_subgroup_accuracy(gt_map, predictions, gt_validity=True, gt_plausibility=True)
        acc_implausible_valid, _, _ = calculate_subgroup_accuracy(gt_map, predictions, gt_validity=True, gt_plausibility=False)
        acc_plausible_invalid, _, _ = calculate_subgroup_accuracy(gt_map, predictions, gt_validity=False, gt_plausibility=True)
        acc_implausible_invalid, _, _ = calculate_subgroup_accuracy(gt_map, predictions, gt_validity=False, gt_plausibility=False)

        conditional_accuracies = {
            'acc_plausible_valid': acc_plausible_valid,
            'acc_implausible_valid': acc_implausible_valid,
            'acc_plausible_invalid': acc_plausible_invalid,
            'acc_implausible_invalid': acc_implausible_invalid
        }

        # --- 4. Calculate Content Effect Bias Metrics ---
        bias_metrics = calculate_content_effect_bias(conditional_accuracies)
        tot_content_effect = bias_metrics['tot_content_effect']

        # --- 5. Calculate Combined Metric ---
        combined_smooth_score = calculate_smooth_combined_metric(overall_acc, tot_content_effect)

        # --- 6. Prepare Final Output Dictionary ---
        final_results = {
            'accuracy': round(overall_acc, 4),
            'content_effect': round(tot_content_effect, 4),
            'combined_score': round(combined_smooth_score, 4)
        }

    # --- 7. Write Results ---
    try:
        with open(output_path, 'w') as f:
            json.dump(final_results, f)
            print(f"Scoring complete. Results written to {output_path}")
    except Exception as e:
        print(f"Error writing final results to file: {e}", file=sys.stderr)


In [ ]:
#@title s2 scoring

import json
import math
import sys
from typing import List, Dict, Any, Tuple, Optional

# --- 1. CORE FUNCTIONS ---

def calculate_f1_premises(gt_map: Dict[str, Any], predictions: List[Dict[str, Any]]) -> float:
    """
    Calculates the Macro-Averaged F1-Score for premise retrieval across all data points.
    (Your original function logic here...)
    """
    total_precision = 0.0
    total_recall = 0.0
    valid_count = 0

    for pred_item in predictions:
        item_id = pred_item['id']
        if item_id in gt_map and 'relevant_premises' in gt_map[item_id] and 'relevant_premises' in pred_item:

            true_positives = set(gt_map[item_id]['relevant_premises'])
            predicted_positives = set(pred_item['relevant_premises'])

            if len(true_positives) == 0:
                continue

            TP = len(true_positives.intersection(predicted_positives))
            FP = len(predicted_positives.difference(true_positives))
            FN = len(true_positives.difference(predicted_positives))

            precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
            recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0

            total_precision += precision
            total_recall += recall
            valid_count += 1

    if valid_count == 0:
        return 0.0

    macro_precision = total_precision / valid_count
    macro_recall = total_recall / valid_count

    f1_score = 2 * (macro_precision * macro_recall) / (macro_precision + macro_recall) if (macro_precision + macro_recall) > 0 else 0.0

    return f1_score * 100

# functions for validity accuracy and bias calculation
def calculate_accuracy(
    ground_truth_list: List[Dict[str, Any]],
    predictions_list: List[Dict[str, Any]],
    metric_name: str,
    prediction_key: str,
    plausibility_filter: Optional[bool] = None
) -> Tuple[float, int, int]:

    gt_map = {item['id']: item for item in ground_truth_list}
    correct_predictions = 0
    total_predictions = 0
    for pred_item in predictions_list:
        item_id = pred_item['id']
        if item_id in gt_map:
            gt_item = gt_map[item_id]
            gt_plausibility = gt_item.get('plausibility')
            if plausibility_filter is not None and gt_plausibility != plausibility_filter:
                continue
            if metric_name in gt_item and prediction_key in pred_item:
                true_label = gt_item[metric_name]
                predicted_label = pred_item[prediction_key]
                if isinstance(true_label, bool) and isinstance(predicted_label, bool):
                    total_predictions += 1
                    if true_label == predicted_label:
                        correct_predictions += 1
    if total_predictions == 0:
        return 0.0, 0, 0
    accuracy = (correct_predictions / total_predictions) * 100
    return accuracy, correct_predictions, total_predictions

def calculate_subgroup_accuracy(
    gt_map: Dict[str, Any],
    predictions_list: List[Dict[str, Any]],
    gt_validity: bool,
    gt_plausibility: bool
) -> Tuple[float, int, int]:

    correct_predictions = 0
    total_predictions = 0
    for pred_item in predictions_list:
        item_id = pred_item['id']
        if item_id in gt_map:
            gt_item = gt_map[item_id]
            if gt_item.get('validity') == gt_validity and gt_item.get('plausibility') == gt_plausibility:
                if 'validity' in gt_item and 'validity' in pred_item:
                    true_label = gt_item['validity']
                    predicted_label = pred_item['validity']
                    if isinstance(true_label, bool) and isinstance(predicted_label, bool):
                        total_predictions += 1
                        if true_label == predicted_label:
                            correct_predictions += 1
    if total_predictions == 0:
        return 0.0, 0, 0
    accuracy = (correct_predictions / total_predictions) * 100
    return accuracy, correct_predictions, total_predictions

def calculate_content_effect_bias(accuracies: Dict[str, float]) -> Dict[str, float]:
    """
    Calculates the content effect bias metrics as defined.
    """

    acc_plausible_valid = accuracies.get('acc_plausible_valid', 0.0)
    acc_implausible_valid = accuracies.get('acc_implausible_valid', 0.0)
    acc_plausible_invalid = accuracies.get('acc_plausible_invalid', 0.0)
    acc_implausible_invalid = accuracies.get('acc_implausible_invalid', 0.0)

    # 1. content_effect_intra_validity_label
    intra_valid_diff = abs(acc_plausible_valid - acc_implausible_valid)
    intra_invalid_diff = abs(acc_plausible_invalid - acc_implausible_invalid)
    content_effect_intra_validity_label = (intra_valid_diff + intra_invalid_diff) / 2.0

    # 2. content_effect_inter_validity_label
    inter_plausible_diff = abs(acc_plausible_valid - acc_plausible_invalid)
    inter_implausible_diff = abs(acc_implausible_valid - acc_implausible_invalid)
    content_effect_inter_validity_label = (inter_plausible_diff + inter_implausible_diff) / 2.0

    # 3. tot_content_effect
    tot_content_effect = (content_effect_intra_validity_label + content_effect_inter_validity_label) / 2.0

    return {
        'content_effect_intra_validity_label': content_effect_intra_validity_label,
        'content_effect_inter_validity_label': content_effect_inter_validity_label,
        'tot_content_effect': tot_content_effect
    }

def calculate_smooth_combined_metric(overall_metric: float, total_content_effect: float) -> float:
    """
    Computes the smoother combined score using the natural logarithm.
    Formula: overall_metric / (1 + ln(1 + content_effect))
    """
    if total_content_effect < 0:
        return 0.0

    log_penalty = math.log(1 + total_content_effect)
    smoother_score = overall_metric / (1 + log_penalty)
    return smoother_score

def run_full_scoring(reference_data_path: str, prediction_path: str, output_path: str):
    """
    Runs the full analysis pipeline for a single submission and writes results to the output path.
    """

    try:
        # 1. Load data from the provided file paths
        with open(reference_data_path, 'r') as f:
            ground_truth = json.load(f)

        with open(prediction_path, 'r') as f:
            predictions = json.load(f)

        # Check that the examples in ground_truth are all covered in predictions
        gt_ids = set([example["id"] for example in ground_truth])
        pd_ids = set([example["id"] for example in predictions])
        diff = len(gt_ids.difference(pd_ids))

        if diff != 0:
            print(f"Error: not all the examples in ground truth have a corresponding prediction", file=sys.stderr)
            final_results = {'accuracy': 0.0, 'f1_premises': 0.0, 'combined_score': 0.0}
            # --- Write Results ---
            try:
                with open(output_path, 'w') as f:
                    json.dump(final_results, f)
                    print(f"Scoring complete. Results written to {output_path}")
            except Exception as e:
                print(f"Error writing final results to file: {e}", file=sys.stderr)

            return

    except FileNotFoundError as e:
        print(f"Error: Required file not found. {e}", file=sys.stderr)
        final_results = {'accuracy': 0.0, 'f1_premises': 0.0, 'combined_score': 0.0}
    except json.JSONDecodeError as e:
        print(f"Error: Invalid JSON format in submission or reference file. {e}", file=sys.stderr)
        final_results = {'accuracy': 0.0, 'f1_premises': 0.0, 'combined_score': 0.0}
    except Exception as e:
        print(f"An unexpected error occurred during file loading: {e}", file=sys.stderr)
        final_results = {'accuracy': 0.0, 'f1_premises': 0.0, 'combined_score': 0.0}

    else:
        gt_map = {item['id']: item for item in ground_truth}

        # --- 2. Calculate F1 and Accuracy ---
        f1_premises = calculate_f1_premises(gt_map, predictions)

        common_args = {
            'ground_truth_list': ground_truth,
            'predictions_list': predictions,
            'metric_name': 'validity',
            'prediction_key': 'validity'
        }
        overall_acc, _, _ = calculate_accuracy(**common_args, plausibility_filter=None)

        # --- 3. Calculate Bias Metrics ---
        acc_plausible_valid, _, _ = calculate_subgroup_accuracy(gt_map, predictions, gt_validity=True, gt_plausibility=True)
        acc_implausible_valid, _, _ = calculate_subgroup_accuracy(gt_map, predictions, gt_validity=True, gt_plausibility=False)
        acc_plausible_invalid, _, _ = calculate_subgroup_accuracy(gt_map, predictions, gt_validity=False, gt_plausibility=True)
        acc_implausible_invalid, _, _ = calculate_subgroup_accuracy(gt_map, predictions, gt_validity=False, gt_plausibility=False)

        conditional_accuracies = {
            'acc_plausible_valid': acc_plausible_valid, 'acc_implausible_valid': acc_implausible_valid,
            'acc_plausible_invalid': acc_plausible_invalid, 'acc_implausible_invalid': acc_implausible_invalid
        }

        bias_metrics = calculate_content_effect_bias(conditional_accuracies)
        tot_content_effect = bias_metrics['tot_content_effect']

        # --- 4. Calculate Combined Metric ---
        overall_performance_metric = (overall_acc + f1_premises) / 2.0
        smoother_score = calculate_smooth_combined_metric(overall_performance_metric, tot_content_effect)

        # --- 5. Prepare Final Output Dictionary (Keys MUST match bundle.yaml) ---
        final_results = {
            'accuracy': round(overall_acc, 4),
            'f1_premises': round(f1_premises, 4),
            'content_effect': round(tot_content_effect, 4),
            'combined_score': round(smoother_score, 4)
        }

    # --- 6. Write Results ---
    try:
        with open(output_path, 'w') as f:
            json.dump(final_results, f)
            print(f"Scoring complete. Results written to {output_path}")
    except Exception as e:
        print(f"Error writing final results to file: {e}", file=sys.stderr)


In [ ]:
run_full_scoring("train_data.json", "predictions.json", "score.json")

Scoring complete. Results written to rezz.json
